[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/attention_lstm.ipynb)

# Attention without a decoder: attention pooling for sentiment

In the `attention_mechanism` notebook, attention helped a **decoder** look back at the encoder states. But attention is more general: it is a way of computing a **weighted average of vectors, where the weights depend on how relevant each vector is**. Here we use it to classify movie reviews as positive or negative.

A recurrent network (here a GRU, a simpler cousin of the LSTM) reads the sentence and produces one hidden state $a^{\langle t' \rangle}$ per word. To classify, we need **one** vector for the whole sentence. Two options:

- **last state**: use the state at the end of the sentence (the same bottleneck as the classic seq2seq);
- **attention pooling**: a weighted average of *all* the states, with weights computed by attention.

There is no decoder, so who asks *"where should I look?"*: what is the **query**? It is a **learned vector** $w$, the same for every sentence. Training turns it into the question *"which words decide the sentiment?"*. The three steps are the same as before:

$$e^{\langle t' \rangle} = w \cdot a^{\langle t' \rangle}, \qquad \alpha^{\langle t' \rangle} = \frac{\exp(e^{\langle t' \rangle})}{\sum_{k=1}^{T_x} \exp(e^{\langle k \rangle})}, \qquad c = \sum_{t'=1}^{T_x} \alpha^{\langle t' \rangle} a^{\langle t' \rangle}$$

and then a linear layer classifies $c$.

| symbol | meaning | in the code |
|---|---|---|
| $a^{\langle t' \rangle}$ | GRU hidden state at word $t'$ | `a[:, t_prime]` |
| $w$ | learned query vector | `self.query` |
| $e^{\langle t' \rangle}$ | score of word $t'$ | `e` |
| $\alpha^{\langle t' \rangle}$ | attention weight of word $t'$ | `alpha` |
| $c$ | sentence vector (context) | `c` |

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=3, sci_mode=False)
device = torch.device("cpu")   # small model and dataset: the CPU is enough (training takes a few seconds)

## The dataset

The same movie reviews used in the `word_embeddings_sentiment` notebook, with four splits:

| split | content |
|---|---|
| `train` | 400 sentences, 10 positive and 10 negative adjectives |
| `test_seen` | 100 new sentences with the **same** adjectives |
| `test_unseen` | 100 sentences with adjectives that **never appear** in `train` (e.g. *superb*, *dreadful*) |
| `negation` | 20 sentences where a negation flips the meaning (e.g. *not bad at all*) |

All sentences use the same templates and nouns for both classes, so the adjective is the only cue.

In [ ]:
url = "https://raw.githubusercontent.com/xiptos/is_notes/refs/heads/main/datasets/movie_sentiment.csv"
df = pd.read_csv(url)

print(df.groupby(["split", "label"]).size().unstack())
df.sample(8, random_state=0)

### From words to ids

The vocabulary is built **only from the training sentences**. Words never seen in training become `<unk>` (unknown), and `<pad>` fills the shorter sentences of a batch.

In [ ]:
def tokenize(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

PAD, UNK = 0, 1
vocab = {"<pad>": PAD, "<unk>": UNK}
for sentence in df[df.split == "train"].sentence:
    for w in tokenize(sentence):
        vocab.setdefault(w, len(vocab))

def encode(sentences):
    ids = [[vocab.get(w, UNK) for w in tokenize(s)] for s in sentences]
    x = torch.full((len(ids), max(map(len, ids))), PAD)
    for i, row in enumerate(ids):
        x[i, :len(row)] = torch.tensor(row)
    return x

print("vocabulary:", len(vocab), "words")
print(tokenize("Honestly, the film was superb"), "->", encode(["Honestly, the film was superb"]).tolist())

**Predict first:** *superb* is not in the training vocabulary (it became id 1, `<unk>`). What will the model be able to do with it?

## The model

- An **embedding** turns each word id into a vector.
- A **bidirectional GRU** reads the sentence left-to-right and right-to-left; $a^{\langle t' \rangle}$ is the concatenation of both directions, so each state knows the words **before and after** $t'$.
- A **pooling** step turns the $T_x$ states into one vector $c$: the last states, or attention.
- A **linear layer** classifies $c$ (2 classes).

The sentences of a batch have different lengths. `pack_padded_sequence` makes the GRU skip the padding, so the "last state" is really the state at the last word of each sentence. For attention, the padding positions are **masked**: their score is set to $-\infty$, so their weight is $\exp(-\infty) = 0$.

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, pooling, emb_dim=32, hid_dim=32, bidirectional=True):
        super().__init__()
        self.pooling = pooling
        self.embedding = nn.Embedding(len(vocab), emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True, bidirectional=bidirectional)
        state_dim = hid_dim * (2 if bidirectional else 1)
        # The query w. A Linear layer without bias computes w · a for every position.
        # (A bias would add the same constant to every score: the softmax would not change.)
        self.query = nn.Linear(state_dim, 1, bias=False)
        self.classifier = nn.Linear(state_dim, 2)

    def forward(self, x):
        mask = x != PAD                                       # (B, Tx)
        lengths = mask.sum(dim=1)
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(x), lengths,
                                                   batch_first=True, enforce_sorted=False)
        packed_a, h_last = self.rnn(packed)
        a, _ = nn.utils.rnn.pad_packed_sequence(packed_a, batch_first=True, total_length=x.size(1))
        # a: (B, Tx, state_dim), all the hidden states a<t'>

        if self.pooling == "last":
            # Last state of each direction (forward: last word, backward: first word)
            c = torch.cat(list(h_last), dim=1)                # (B, state_dim)
            alpha = None
        else:
            e = self.query(a).squeeze(2)                      # (B, Tx)   scores e<t'> = w · a<t'>
            e = e.masked_fill(~mask, float("-inf"))           # no attention to the padding
            alpha = F.softmax(e, dim=1)                       # (B, Tx)   weights, sum to 1
            c = torch.bmm(alpha.unsqueeze(1), a).squeeze(1)   # (B, state_dim)   weighted average

        return self.classifier(c), alpha

## Training

The two models are trained in the same way: 20 epochs, mini-batches of 32 sentences, cross-entropy loss.

In [ ]:
def train(model, epochs=20, batch_size=32, seed=0):
    torch.manual_seed(seed)
    d = df[df.split == "train"]
    X, y = encode(d.sentence), torch.tensor(d.label.values)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    for epoch in range(1, epochs + 1):
        perm = torch.randperm(len(X))
        total = 0.0
        for i in range(0, len(X), batch_size):
            batch = perm[i:i + batch_size]
            logits, _ = model(X[batch])
            loss = F.cross_entropy(logits, y[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(batch)
        if epoch % 5 == 0:
            print(f"epoch {epoch:2d}  loss {total / len(X):.4f}")
    return model

@torch.no_grad()
def accuracy(model, split):
    d = df[df.split == split]
    logits, _ = model(encode(d.sentence))
    return (logits.argmax(dim=1).numpy() == d.label.values).mean()

torch.manual_seed(0)
model_last = train(SentimentRNN("last"))
torch.manual_seed(0)
model_attention = train(SentimentRNN("attention"))

for name, m in [("last state", model_last), ("attention", model_attention)]:
    print(f"{name:>10s}: test_seen accuracy = {accuracy(m, 'test_seen'):.2f}")

Both models are perfect on new sentences with known adjectives. With sentences of 3 to 9 words, the last states of a bidirectional GRU are not a real bottleneck. So what does attention add here? It tells us **where the model looks**.

## Where does the model look?

Each word is highlighted with its attention weight $\alpha^{\langle t' \rangle}$ (darker = more attention). Words marked with `*` are not in the training vocabulary: the model only sees `<unk>`. The number on the left is the predicted probability of *positive*.

In [ ]:
@torch.no_grad()
def show_attention(model, sentences, title=""):
    logits, alpha = model(encode(sentences))
    p_positive = logits.softmax(dim=1)[:, 1]
    fig, ax = plt.subplots(figsize=(10, 0.42 * len(sentences) + 0.6))
    cmap = plt.get_cmap("Oranges")
    for row, (sentence, weights, p) in enumerate(zip(sentences, alpha, p_positive)):
        y = len(sentences) - row
        ax.text(0, y, f"{p:.2f}", ha="left", va="center", family="monospace",
                color="tab:green" if p > 0.5 else "tab:red", fontweight="bold")
        x = 0.07
        for w, a_w in zip(tokenize(sentence), weights.tolist()):
            label = w if w in vocab else w + "*"
            t = ax.text(x, y, label, ha="left", va="center", fontsize=11,
                        bbox=dict(facecolor=cmap(a_w), edgecolor="none", boxstyle="round,pad=0.25"),
                        color="white" if a_w > 0.6 else "black")
            fig.canvas.draw()
            bb = t.get_window_extent().transformed(ax.transData.inverted())
            x = bb.x1 + 0.012
    ax.set_xlim(0, 1)
    ax.set_ylim(0.4, len(sentences) + 0.6)
    ax.axis("off")
    ax.set_title(title, loc="left")
    plt.show()

examples = ["It was a good performance",
            "Honestly, the performance was amazing",
            "What a disappointing plot",
            "My friends found the script horrible",
            "In my opinion, the soundtrack was really enjoyable"]
show_attention(model_attention, examples, "Attention pooling: known adjectives")

**Explain:**

- Nobody told the model which words are adjectives. How did the query $w$ learn to point at them?
- In some sentences (e.g. *My friends found the script horrible*) the weight is spread over several words. The weights are computed from the states $a^{\langle t' \rangle}$, not from the words: with a bidirectional GRU, the state of every word also contains information about its neighbours, so several positions can be equally useful.

## Experiment 1: adjectives never seen in training

**Predict first:** the adjectives of `test_unseen` are all `<unk>`. What accuracy do you expect? Where will the attention go?

In [ ]:
for name, m in [("last state", model_last), ("attention", model_attention)]:
    print(f"{name:>10s}: test_unseen accuracy = {accuracy(m, 'test_unseen'):.2f}")

show_attention(model_attention,
               ["The ending felt tedious to me", "Honestly, the film was superb",
                "Overall, a lousy soundtrack", "It was a magnificent story"],
               "Attention pooling: unknown adjectives")

**Explain:** the attention can still find **where** the adjective is (from the position and the words around it), but every unknown adjective is the same `<unk>` vector, so the model cannot know **what** it says. Attention selects information; it cannot create it. (In the `word_embeddings_sentiment` notebook, pre-trained GloVe vectors solved this problem.)

## Experiment 2: negation

**Predict first:** *not* never appears in the training sentences. How will the models handle *The film was not bad at all*?

In [ ]:
for name, m in [("last state", model_last), ("attention", model_attention)]:
    print(f"{name:>10s}: negation accuracy = {accuracy(m, 'negation'):.2f}")

negation = df[df.split == "negation"]
show_attention(model_attention, negation.sentence.tolist()[:8], "Attention pooling: negation")
negation.head(8)[["sentence", "label"]]

**Explain:**

- The accuracy is far **below** 50%: the models are not guessing, they are systematically wrong. Why?
- Look at the weights: the model still looks at the adjective (*bad*, *excellent*, ...). Is *not* even a word for this model?
- A high attention weight means *"this position was used"*, not *"the model understood this word"*. Be careful when you read attention weights as explanations.

## Connection with the seq2seq attention

| | seq2seq (`attention_mechanism`) | attention pooling (this notebook) |
|---|---|---|
| vectors averaged | encoder states $a^{\langle t' \rangle}$ | GRU states $a^{\langle t' \rangle}$ |
| query | decoder state $s^{\langle t \rangle}$, changes at every step | learned vector $w$, the same for every sentence |
| attention distributions per sentence | one per output word ($T_y$ rows) | one |
| score | dot, general or additive | dot with $w$ |

Transformers go one step further with **self-attention**: every word is also a query, so each word computes its own weighted average of all the words of the sentence.

## Your turn

1. **The bias does nothing.** Change `bias=False` to `bias=True` in `self.query`, retrain, and compare the weights. Explain with the shift-invariance of the softmax.
2. **Unidirectional GRU.** Train `SentimentRNN("attention", bidirectional=False)` and compare the highlights, e.g. for *It was a good performance*. With a left-to-right GRU, the state at *performance* has already read *good*. Does part of the attention move to the end of the sentence?
3. **Teach it negation.** Write 20–40 training sentences with *not*, *never*, *far from* (both classes) and add them to `train`. Does the negation accuracy improve? Where does the attention go now?
4. **Two queries.** Replace `nn.Linear(state_dim, 1)` by `nn.Linear(state_dim, 2)`, compute one $\alpha$ and one $c$ per query and concatenate the two context vectors before the classifier. Do the two queries look at different words? (This is the idea behind **multi-head** attention.)
5. **Pre-trained embeddings.** Initialize `self.embedding` with the GloVe vectors from `word_embeddings_sentiment` (frozen), building the vocabulary from all the words in GloVe. Does `test_unseen` improve?

In [ ]:
# Your turn